# Setup SQLite : tout en local, sans serveur

SQLite est embarqué dans la bibliothèque standard de Python via le module `sqlite3`. Il n'y a rien à installer, pas de serveur à démarrer, pas de credentials. La base est un simple fichier `.sqlite` (ou `.db`) sur le disque.

Quand SQLite est pratique :
- prototyper rapidement, tester une requête
- partager un exercice avec ses données dans un seul fichier
- éviter d'installer PostgreSQL juste pour s'entraîner

Quand SQLite n'est pas adapté :
- accès concurrent en écriture par plusieurs clients (un seul writer à la fois)
- déploiement multi-utilisateurs en production
- besoin de types riches (ARRAY, JSON natif, HSTORE, INTERVAL, UUID, etc.)


## Vérification de la version

In [1]:
import sqlite3
print(sqlite3.sqlite_version)

3.43.2


## Connexion

Deux modes :
- fichier sur disque : `sqlite3.connect("ma_base.sqlite")`. Le fichier est créé s'il n'existe pas.
- en mémoire : `sqlite3.connect(":memory:")`. Rien n'est persisté, utile pour les tests.


In [60]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("SELECT sqlite_version()")
print(cur.fetchone())

('3.43.2',)


## Créer une table, insérer, lire

Les placeholders sont `?` (pas `%s` comme avec psycopg2).

## Séries : 

- Créer une table `series` 
    - Avec une colonne d'id (comme ci-dessous) : `id`
        - `INTEGER PRIMARY KEY AUTOINCREMENT`
    - Le nom de la série : `name` ou `title`
        - TEXT NOT NULL
    - Le nom du réalisateur : `director`
        - TEXT NOT NULL
    - La date de sortie (date du premier épisode de la saison 1) : `air_date` ou `release_date`
        - Faire une petite recherche pour voir comment stocker une date
        - Décider si vous voulez qu'elle puisse être nulle ou non
    - La note de la série (IMDB / Senscritique / Allociné) : `rating`
        - REAL (vous pouvez faire une petite recherche pour trouver un type précis)
     
- Insérer les données de 2-5 séries
    - Wikipedia en one shot très probable
 
- Afficher toutes les lignes de la table

(1, 'Game of Thrones', 'Some director', '2011-04-17', 5.0, None)
(2, 'Friends', 'Some american person', '1994-01-01', 7.5, None)


(1, 'Game of Thrones', 'Some director', '2011-04-17', 5.0, '2020-12-31')
(2, 'Friends', 'Some american person', '1994-01-01', 7.5, '2004-12-31')


(1, 'Game of Thrones', 'Some director', '2011-04-17', 5.0, '2020-12-31', 3546)
(2, 'Friends', 'Some american person', '1994-01-01', 7.5, '2004-12-31', 4017)


In [ ]:
# cur.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    price REAL,
    in_stock INTEGER
)
""")

cur.executemany(
    "INSERT INTO products (name, price, in_stock) VALUES (?, ?, ?)",
    [("Pomme", 0.50, 100), ("Banane", 0.30, 150), ("Orange", 0.40, 75)],
)
conn.commit()

for row in cur.execute("SELECT * FROM products ORDER BY name DESC"):
    print(row)

## Charger Northwind à partir du dump PostgreSQL

Le dump `data/northwind.sql` a été généré par `pg_dump`. Pour le charger sous SQLite il faut adapter quelques éléments propres à PostgreSQL :

- retirer les directives `SET ...;` (paramètres de session psql)
- retirer les `ALTER TABLE ONLY ... ADD CONSTRAINT ...` (SQLite ne permet pas d'ajouter une contrainte FK après création de la table)
- remplacer le type `bytea` par `BLOB`
- remplacer le littéral `'\x'` (bytea vide) par `NULL`

Cette adaptation suffit pour charger les 14 tables et exécuter les requêtes `SELECT` du cours.

In [71]:
import os
import re
import sqlite3

DUMP = "../../data/northwind.sql"
DB = "northwind.sqlite"

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

if os.path.exists(DB):
    os.remove(DB)

conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()

cur = conn.cursor()
for table in ["customers", "orders", "order_details", "products", "employees"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]}")

customers: 91
orders: 830
order_details: 2155
products: 77
employees: 9


## Quelques requêtes pour vérifier

In [5]:
# SELECT simple
for row in cur.execute("SELECT contact_name, city FROM customers LIMIT 5"):
    print(row)

('Maria Anders', 'Berlin')
('Ana Trujillo', 'México D.F.')
('Antonio Moreno', 'México D.F.')
('Thomas Hardy', 'London')
('Christina Berglund', 'Luleå')


In [6]:
# JOIN + GROUP BY
query = """
SELECT c.country, COUNT(o.order_id) AS nb_commandes
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY nb_commandes DESC
LIMIT 5
"""
for row in cur.execute(query):
    print(row)

('USA', 122)
('Germany', 122)
('Brazil', 83)
('France', 77)
('UK', 56)


## Ce qui ne fonctionne pas (ou diffère de PostgreSQL)

Plusieurs constructions vues dans les TP PostgreSQL ne passent pas tel quel sous SQLite. Vue d'ensemble.

### Métacommandes psql

`\dt`, `\d table`, `\c base`, `\l` ne sont pas du SQL : ce sont des commandes du client `psql`. Équivalents SQLite :

```sql
-- lister les tables
SELECT name FROM sqlite_master WHERE type = 'table';

-- structure d'une table
PRAGMA table_info(orders);

-- index d'une table
PRAGMA index_list(orders);

-- contraintes FK d'une table
PRAGMA foreign_key_list(orders);
```

### Types absents

| PostgreSQL | SQLite |
|---|---|
| `SERIAL`, `BIGSERIAL` | `INTEGER PRIMARY KEY AUTOINCREMENT` |
| `GENERATED BY DEFAULT AS IDENTITY` | idem |
| `CREATE SEQUENCE ...` | pas supporté (utiliser AUTOINCREMENT) |
| `BOOLEAN` | stocké en `INTEGER` 0/1 (`TRUE`/`FALSE` acceptés et convertis) |
| `DATE`, `TIMESTAMP`, `TIMESTAMPTZ` | pas de type natif, stocker en TEXT ISO 8601 ou REAL (julian) |
| `INTERVAL` | non supporté |
| `UUID`, `HSTORE`, `INET`, `CIDR` | non natifs |
| `ARRAY` | non supporté (utiliser une table de liaison ou du JSON) |
| `JSON`, `JSONB` | pas de type, mais module JSON1 disponible (`json_extract(...)`) |

### Jointures

- `FULL OUTER JOIN` et `RIGHT JOIN` : supportés seulement à partir de SQLite 3.39 (octobre 2022). Vérifier avec `sqlite3.sqlite_version`.

### Agrégation avancée

- `GROUPING SETS`, `CUBE`, `ROLLUP` : non supportés. Contournement : plusieurs `GROUP BY` chaînés avec `UNION ALL`.
- `DISTINCT ON (col)` : non supporté. Contournement avec une fonction de fenêtre :

```sql
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ship_name ORDER BY order_date) AS rn
  FROM orders
) WHERE rn = 1;
```

### Recherche insensible à la casse

`ILIKE` n'existe pas. Trois options :

```sql
SELECT * FROM customers WHERE contact_name LIKE 'a%' COLLATE NOCASE;
SELECT * FROM customers WHERE LOWER(contact_name) LIKE 'a%';
SELECT * FROM customers WHERE contact_name GLOB '[Aa]*';
```

### Modifications de schéma (ALTER TABLE)

SQLite a un `ALTER TABLE` très réduit :
- `ADD COLUMN` : OK
- `RENAME TO` / `RENAME COLUMN` : OK
- `DROP COLUMN` : OK depuis 3.35 (mars 2021)
- `ALTER COLUMN ... TYPE ...` : **non supporté**. Il faut recréer la table.
- `ADD CONSTRAINT` après coup : **non supporté**. Toutes les contraintes (PK, FK, CHECK, UNIQUE) doivent être dans le `CREATE TABLE` initial.

### Fonctions et triggers

- Pas de PL/pgSQL. La syntaxe `CREATE OR REPLACE FUNCTION ... LANGUAGE plpgsql` n'existe pas.
- On enregistre une fonction Python pour qu'elle soit appelable en SQL :

```python
def total_score(user_id, exercice_id):
    # logique métier en Python
    return 42

conn.create_function("total_score", 2, total_score)
cur.execute("SELECT total_score(1, 1)").fetchone()
```

- Les triggers existent mais sans `EXECUTE FUNCTION` : on met directement le corps `BEGIN ... END`.

```sql
CREATE TRIGGER after_answers_insert
AFTER INSERT ON answers
FOR EACH ROW
BEGIN
    UPDATE exercice_stats
       SET cumulative_score = cumulative_score + CASE WHEN NEW.is_correct THEN 1 ELSE 0 END
     WHERE exercice_id = NEW.exercice_id AND user_id = NEW.user_id;
END;
```

### Vues

Les vues `CREATE VIEW` fonctionnent en lecture. Pour modifier les données à travers la vue, il faut un trigger `INSTEAD OF` (PostgreSQL gère certaines vues comme directement modifiables).

### Plan d'exécution

`EXPLAIN ANALYZE` n'existe pas. Équivalent :

```sql
EXPLAIN QUERY PLAN
SELECT * FROM orders WHERE ship_country = 'France';
```

### Transactions

`BEGIN` / `COMMIT` / `ROLLBACK` existent. Différence : sous `sqlite3`, la connexion gère les transactions implicitement. Pour un contrôle propre, utiliser `conn` comme context manager :

```python
with conn:
    conn.execute("INSERT INTO ...")
    conn.execute("UPDATE ...")
# commit auto à la sortie du bloc, rollback si exception
```

### Concurrence

Un seul writer simultané (verrou au niveau du fichier). Plusieurs readers OK. Pour de la prod multi-clients, PostgreSQL reste le bon choix.


## Fermeture

In [7]:
conn.close()